In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import tarfile

tar_path = "/content/drive/My Drive/AIO_Homework/yolov1/data/VOCtrainval_11-May-2012.tar"
extract_dir = "/content/data/"
voc2012_dir = os.path.join(extract_dir, "VOCdevkit", "VOC2012")

if not os.path.exists(voc2012_dir):
    print(f"Đang giải nén {tar_path}...")
    with tarfile.open(tar_path, 'r') as tar:
        tar.extractall(path=extract_dir)
    print("Giải nén hoàn tất!")
else:
    print(f"Thư mục dữ liệu đã tồn tại: {voc2012_dir}")

Đang giải nén /content/drive/My Drive/AIO_Homework/yolov1/data/VOCtrainval_11-May-2012.tar...


/tmp/ipykernel_789/1399107315.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Giải nén hoàn tất!


In [ ]:
import os
import torch
import xml.etree.ElementTree as ET
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

class VOCDataset(Dataset):
    def __init__(self, root_dir, S=7, B=2, C=20, transform=None):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "JPEGImages")
        self.ann_dir = os.path.join(root_dir, "Annotations")
        self.S = S
        self.B = B
        self.C = C
        self.transform = transform

        self.class_to_idx = {c: i for i, c in enumerate(VOC_CLASSES)}
        self.image_ids = [f.split('.')[0] for f in os.listdir(self.img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        img_id = self.image_ids[index]
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        xml_path = os.path.join(self.ann_dir, f"{img_id}.xml")

        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []
        for obj in root.findall("object"):
            cls_name = obj.find("name").text
            if cls_name not in self.class_to_idx:
                continue
            label = self.class_to_idx[cls_name]

            bndbox = obj.find("bndbox")
            xmin = float(bndbox.find("xmin").text) / orig_w
            ymin = float(bndbox.find("ymin").text) / orig_h
            xmax = float(bndbox.find("xmax").text) / orig_w
            ymax = float(bndbox.find("ymax").text) / orig_h

            x_center = (xmin + xmax) / 2.0
            y_center = (ymin + ymax) / 2.0
            w = xmax - xmin
            h = ymax - ymin

            boxes.append([x_center, y_center, w, h])
            labels.append(label)

        if self.transform:
            image = self.transform(image)

        target = torch.zeros((self.S, self.S, self.C + 5))

        for box, label in zip(boxes, labels):
            x, y, w, h = box

            grid_i = int(self.S * y)
            grid_j = int(self.S * x)

            x_cell = self.S * x - grid_j
            y_cell = self.S * y - grid_i

            if target[grid_i, grid_j, self.C + 4] == 0:
                target[grid_i, grid_j, label] = 1.0
                target[grid_i, grid_j, self.C:self.C + 4] = torch.tensor([x_cell, y_cell, w, h])
                target[grid_i, grid_j, self.C + 4] = 1.0

        return image, target

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ModularBackbone(nn.Module):
    """
    Modular Feature Extractor hỗ trợ 6 loại backbone theo đề bài:
    - custom_cnn (YOLOv1 gốc)
    - vgg16
    - resnet18
    - resnet50
    - efficientnet_b0
    - mobilenet_v2
    """
    def __init__(self, backbone_name='custom_cnn', pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name.lower()

        if self.backbone_name == 'custom_cnn':
            self.features = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
                nn.BatchNorm2d(64),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2, 2),

                nn.Conv2d(64, 192, kernel_size=3, padding=1),
                nn.BatchNorm2d(192),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2, 2),

                nn.Conv2d(192, 256, kernel_size=3, padding=1),
                nn.BatchNorm2d(256),
                nn.LeakyReLU(0.1),
                nn.Conv2d(256, 512, kernel_size=3, padding=1),
                nn.BatchNorm2d(512),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2, 2),

                nn.Conv2d(512, 1024, kernel_size=3, padding=1),
                nn.BatchNorm2d(1024),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2, 2),
            )
            self.out_channels = 1024

        elif self.backbone_name == 'vgg16':
            try:
                vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT if pretrained else None)
            except AttributeError:
                vgg = models.vgg16(pretrained=pretrained)
            self.features = vgg.features
            self.out_channels = 512

        elif self.backbone_name == 'resnet18':
            try:
                resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
            except AttributeError:
                resnet = models.resnet18(pretrained=pretrained)
            self.features = nn.Sequential(*list(resnet.children())[:-2])
            self.out_channels = 512

        elif self.backbone_name == 'resnet50':
            try:
                resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
            except AttributeError:
                resnet = models.resnet50(pretrained=pretrained)
            self.features = nn.Sequential(*list(resnet.children())[:-2])
            self.out_channels = 2048

        elif self.backbone_name == 'efficientnet_b0':
            try:
                effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
            except AttributeError:
                effnet = models.efficientnet_b0(pretrained=pretrained)
            self.features = effnet.features
            self.out_channels = 1280

        elif self.backbone_name == 'mobilenet_v2':
            try:
                mbnet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
            except AttributeError:
                mbnet = models.mobilenet_v2(pretrained=pretrained)
            self.features = mbnet.features
            self.out_channels = 1280

        else:
            raise ValueError(f"Backbone '{backbone_name}' không hợp lệ!")

        self.pool = nn.AdaptiveAvgPool2d((7, 7))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return x


class YOLOv1(nn.Module):
    def __init__(self, backbone_name='resnet18', pretrained=True, S=7, B=2, C=20):
        super().__init__()
        self.S = S
        self.B = B
        self.C = C

        self.backbone = ModularBackbone(backbone_name=backbone_name, pretrained=pretrained)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.backbone.out_channels * S * S, 1024),
            nn.Dropout(0.5),
            nn.LeakyReLU(0.1),
            nn.Linear(1024, S * S * (B * 5 + C))
        )

    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        return out.view(-1, self.S, self.S, self.B * 5 + self.C)

In [ ]:
import torch
import torch.nn as nn

def intersection_over_union(boxes_preds, boxes_labels):
    box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
    box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
    box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
    box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2

    box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
    box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
    box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
    box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    box1_area = torch.abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = torch.abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    return intersection / (box1_area + box2_area - intersection + 1e-6)


class YOLOLoss(nn.Module):
    def __init__(self, S=7, B=2, C=20, lambda_coord=5.0, lambda_noobj=0.5):
        super().__init__()
        self.S = S
        self.B = B
        self.C = C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.mse = nn.MSELoss(reduction="sum")

    def forward(self, predictions, target):
        iou_b1 = intersection_over_union(predictions[..., 20:24], target[..., 20:24])
        iou_b2 = intersection_over_union(predictions[..., 25:29], target[..., 20:24])
        ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim=0)

        best_box = torch.argmax(ious, dim=0)
        exists_box = target[..., 24:25]

        # 1. Box Coordinate Loss (Tạo tensor mới để tránh lỗi In-place operation trong Autograd)
        box_preds = (
            best_box * predictions[..., 25:29] + (1 - best_box) * predictions[..., 20:24]
        )
        box_targs = target[..., 20:24]

        pred_xy = box_preds[..., 0:2]
        pred_wh = torch.sign(box_preds[..., 2:4]) * torch.sqrt(
            torch.abs(box_preds[..., 2:4]) + 1e-6
        )
        box_predictions_transformed = torch.cat([pred_xy, pred_wh], dim=-1)

        targ_xy = box_targs[..., 0:2]
        targ_wh = torch.sqrt(torch.abs(box_targs[..., 2:4]) + 1e-6)
        box_targets_transformed = torch.cat([targ_xy, targ_wh], dim=-1)

        loss_coord = self.mse(
            torch.flatten(exists_box * box_predictions_transformed, end_dim=-2),
            torch.flatten(exists_box * box_targets_transformed, end_dim=-2),
        )

        # 2. Object Loss
        pred_conf = (
            best_box * predictions[..., 29:30] + (1 - best_box) * predictions[..., 24:25]
        )
        loss_obj = self.mse(
            torch.flatten(exists_box * pred_conf),
            torch.flatten(exists_box * target[..., 24:25]),
        )

        # 3. No Object Loss
        loss_noobj = self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 24:25], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 24:25], start_dim=1),
        ) + self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 29:30], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 24:25], start_dim=1),
        )

        # 4. Class Loss
        loss_class = self.mse(
            torch.flatten(exists_box * predictions[..., :20], end_dim=-2),
            torch.flatten(exists_box * target[..., :20], end_dim=-2),
        )

        total_loss = (
            self.lambda_coord * loss_coord
            + loss_obj
            + self.lambda_noobj * loss_noobj
            + loss_class
        )

        return total_loss

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for i, (images, targets) in enumerate(dataloader):
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        predictions = model(images)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        if (i + 1) % 10 == 0 or (i + 1) == len(dataloader):
            print(f"  Batch [{i+1}/{len(dataloader)}] - Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị đang dùng: {device}")

transform = T.Compose([
    T.Resize((448, 448)),
    T.ToTensor(),
])

# Đã sửa lại đường dẫn cho đúng với môi trường Colab
voc_dir = "/content/data/VOCdevkit/VOC2012"
dataset = VOCDataset(root_dir=voc_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

print(f"Tổng số ảnh trong dataset: {len(dataset)}")

# Danh sách 6 backbone theo yêu cầu đề bài
backbones = ['custom_cnn', 'vgg16', 'resnet18', 'resnet50', 'efficientnet_b0', 'mobilenet_v2']

print("\n=== THỬ NGHIỆM TẤT CẢ BACKBONE ===")
for bb in backbones:
    print(f"\n---> Đang thử nghiệm YOLOv1 với Backbone: {bb}")
    model = YOLOv1(backbone_name=bb, pretrained=True).to(device)
    criterion = YOLOLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    loss = train_one_epoch(model, dataloader, optimizer, criterion, device)
    print(f"✅ Hoàn thành 1 Epoch | Backbone: {bb} | Loss: {loss:.4f}")

Thiết bị đang dùng: cuda
Tổng số ảnh trong dataset: 17125

=== THỬ NGHIỆM TẤT CẢ BACKBONE ===

---> Đang thử nghiệm YOLOv1 với Backbone: custom_cnn
  Batch [10/2141] - Loss: 593.1003
  Batch [20/2141] - Loss: 291.6307
  Batch [30/2141] - Loss: 327.5819
  Batch [40/2141] - Loss: 242.0685
  Batch [50/2141] - Loss: 284.4274
  Batch [60/2141] - Loss: 144.0839
  Batch [70/2141] - Loss: 101.0779
  Batch [80/2141] - Loss: 207.0672
  Batch [90/2141] - Loss: 85.3092
  Batch [100/2141] - Loss: 89.4681
  Batch [110/2141] - Loss: 112.1940
  Batch [120/2141] - Loss: 127.8763
  Batch [130/2141] - Loss: 90.0121
  Batch [140/2141] - Loss: 156.1006
  Batch [150/2141] - Loss: 238.1841
  Batch [160/2141] - Loss: 83.4612
  Batch [170/2141] - Loss: 83.8348
  Batch [180/2141] - Loss: 125.9097
  Batch [190/2141] - Loss: 120.1436
  Batch [200/2141] - Loss: 73.2330
  Batch [210/2141] - Loss: 83.6860
  Batch [220/2141] - Loss: 114.5161
  Batch [230/2141] - Loss: 110.7537
  Batch [240/2141] - Loss: 92.2956
  Bat

100%|██████████| 528M/528M [00:07<00:00, 71.6MB/s]


  Batch [10/2141] - Loss: 119.2897
  Batch [20/2141] - Loss: 106.3144
  Batch [30/2141] - Loss: 170.7422
  Batch [40/2141] - Loss: 84.7582
  Batch [50/2141] - Loss: 84.7212
  Batch [60/2141] - Loss: 99.9639
  Batch [70/2141] - Loss: 81.4507
  Batch [80/2141] - Loss: 31.1339
  Batch [90/2141] - Loss: 44.8199
  Batch [100/2141] - Loss: 55.8732
  Batch [110/2141] - Loss: 35.9583
  Batch [120/2141] - Loss: 74.3546
  Batch [130/2141] - Loss: 64.3974
  Batch [140/2141] - Loss: 72.5299
  Batch [150/2141] - Loss: 71.2879
  Batch [160/2141] - Loss: 63.9407
  Batch [170/2141] - Loss: 44.6967
  Batch [180/2141] - Loss: 49.5222
  Batch [190/2141] - Loss: 45.5678
  Batch [200/2141] - Loss: 50.1310
  Batch [210/2141] - Loss: 49.6044
  Batch [220/2141] - Loss: 52.5406
  Batch [230/2141] - Loss: 66.3687
  Batch [240/2141] - Loss: 55.6047
  Batch [250/2141] - Loss: 63.0017
  Batch [260/2141] - Loss: 53.9378
  Batch [270/2141] - Loss: 41.8462
  Batch [280/2141] - Loss: 62.4882
  Batch [290/2141] - Loss:

100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s]


  Batch [10/2141] - Loss: 506.5278
  Batch [20/2141] - Loss: 235.7347
  Batch [30/2141] - Loss: 202.8387
  Batch [40/2141] - Loss: 206.8313
  Batch [50/2141] - Loss: 219.8905
  Batch [60/2141] - Loss: 147.1385
  Batch [70/2141] - Loss: 120.7185
  Batch [80/2141] - Loss: 107.2647
  Batch [90/2141] - Loss: 94.5767
  Batch [100/2141] - Loss: 87.4152
  Batch [110/2141] - Loss: 137.4034
  Batch [120/2141] - Loss: 82.8616
  Batch [130/2141] - Loss: 136.5322
  Batch [140/2141] - Loss: 73.7492
  Batch [150/2141] - Loss: 88.0165
  Batch [160/2141] - Loss: 61.7576
  Batch [170/2141] - Loss: 76.4658
  Batch [180/2141] - Loss: 81.7576
  Batch [190/2141] - Loss: 128.2879
  Batch [200/2141] - Loss: 84.7662
  Batch [210/2141] - Loss: 92.9407
  Batch [220/2141] - Loss: 148.8923
  Batch [230/2141] - Loss: 110.1276
  Batch [240/2141] - Loss: 37.0468
  Batch [250/2141] - Loss: 102.6831
  Batch [260/2141] - Loss: 76.4462
  Batch [270/2141] - Loss: 52.9338
  Batch [280/2141] - Loss: 92.0058
  Batch [290/21

...

  Batch [10/2141] - Loss: 147.7888
  Batch [20/2141] - Loss: 80.5151
  Batch [30/2141] - Loss: 87.5510
  Batch [40/2141] - Loss: 80.5963
  Batch [50/2141] - Loss: 80.1207
  Batch [60/2141] - Loss: 69.9377
  Batch [70/2141] - Loss: 72.7269
  Batch [80/2141] - Loss: 45.2694
  Batch [90/2141] - Loss: 100.8757
  Batch [100/2141] - Loss: 58.3843
  Batch [110/2141] - Loss: 44.1923
  Batch [120/2141] - Loss: 58.1969
  Batch [130/2141] - Loss: 41.0327
  Batch [140/2141] - Loss: 39.4775
  Batch [150/2141] - Loss: 53.7344
  Batch [160/2141] - Loss: 55.6019
  Batch [170/2141] - Loss: 75.0798
  Batch [180/2141] - Loss: 45.2806
  Batch [190/2141] - Loss: 23.7716
  Batch [200/2141] - Loss: 97.4189
  Batch [210/2141] - Loss: 53.0238
  Batch [220/2141] - Loss: 78.7013
  Batch [230/2141] - Loss: 66.2288
  Batch [240/2141] - Loss: 36.0753
  Batch [250/2141] - Loss: 40.1674
  Batch [260/2141] - Loss: 45.4309
  Batch [270/2141] - Loss: 49.7708
  Batch [280/2141] - Loss: 36.6294
  Batch [290/2141] - Loss: 

100%|██████████| 20.5M/20.5M [00:00<00:00, 132MB/s] 


  Batch [10/2141] - Loss: 165.8348
  Batch [20/2141] - Loss: 151.4251
  Batch [30/2141] - Loss: 215.5627
  Batch [40/2141] - Loss: 138.5286
  Batch [50/2141] - Loss: 85.7330
  Batch [60/2141] - Loss: 107.8956
  Batch [70/2141] - Loss: 123.2841
  Batch [80/2141] - Loss: 92.1086
  Batch [90/2141] - Loss: 88.5907
  Batch [100/2141] - Loss: 70.0832
  Batch [110/2141] - Loss: 54.0572
  Batch [120/2141] - Loss: 80.2242
  Batch [130/2141] - Loss: 44.0256
  Batch [140/2141] - Loss: 96.8654
  Batch [150/2141] - Loss: 60.0025
  Batch [160/2141] - Loss: 60.3216
  Batch [170/2141] - Loss: 59.1194
  Batch [180/2141] - Loss: 56.5211
  Batch [190/2141] - Loss: 53.4597
  Batch [200/2141] - Loss: 46.3369
  Batch [210/2141] - Loss: 50.7160
  Batch [220/2141] - Loss: 51.6523
  Batch [230/2141] - Loss: 87.6233
  Batch [240/2141] - Loss: 104.5937
  Batch [250/2141] - Loss: 47.8836
  Batch [260/2141] - Loss: 71.7953
  Batch [270/2141] - Loss: 84.1984
  Batch [280/2141] - Loss: 48.0838
  Batch [290/2141] - L

100%|██████████| 13.6M/13.6M [00:00<00:00, 81.2MB/s]


  Batch [10/2141] - Loss: 172.9167
  Batch [20/2141] - Loss: 117.8457
  Batch [30/2141] - Loss: 91.8414
  Batch [40/2141] - Loss: 108.7401
  Batch [50/2141] - Loss: 102.6056
  Batch [60/2141] - Loss: 50.4750
  Batch [70/2141] - Loss: 80.6853
  Batch [80/2141] - Loss: 63.9213
  Batch [90/2141] - Loss: 89.5291
  Batch [100/2141] - Loss: 58.7543
  Batch [110/2141] - Loss: 36.9822
  Batch [120/2141] - Loss: 59.6309
  Batch [130/2141] - Loss: 47.3388
  Batch [140/2141] - Loss: 101.7725
  Batch [150/2141] - Loss: 40.9813
  Batch [160/2141] - Loss: 45.8807
  Batch [170/2141] - Loss: 55.5209
  Batch [180/2141] - Loss: 72.5711
  Batch [190/2141] - Loss: 95.6817
  Batch [200/2141] - Loss: 66.4487
  Batch [210/2141] - Loss: 53.1067
  Batch [220/2141] - Loss: 92.3865
  Batch [230/2141] - Loss: 34.5300
  Batch [240/2141] - Loss: 45.3950
  Batch [250/2141] - Loss: 65.7961
  Batch [260/2141] - Loss: 79.0841
  Batch [270/2141] - Loss: 49.7157
  Batch [280/2141] - Loss: 64.5345
  Batch [290/2141] - Los